# 01 — Real-Time Athlete Performance Analytics and Anomaly Detection

This notebook runs the reproducible synthetic-data demonstration. Replace synthetic data with authorized sensor records for real evaluation.

In [ ]:
from src.data_generator import generate_demo_data
from src.preprocessing import validate_and_clean
from src.feature_engineering import engineer_features
from src.modeling import train_performance_model, FEATURES
from src.anomaly_detection import fit_anomaly_detector, apply_anomaly_detector
from src.alerts import generate_alerts
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

Path("results").mkdir(exist_ok=True)


In [ ]:
# 1. Generate synthetic demo data (not real athlete measurements)
raw = generate_demo_data(n_athletes=8, samples_per_athlete=600, seed=42)
print(raw.shape, raw.athlete_id.nunique())
display(raw.head())


In [ ]:
# 2. Validate and engineer features
clean = validate_and_clean(raw)
featured = engineer_features(clean)
display(featured.head())


In [ ]:
# 3. Train Random Forest performance estimator
performance_model, metrics, test_idx = train_performance_model(featured)
print("Group-held-out regression metrics:", metrics)


In [ ]:
# 4. Fit Isolation Forest and score all records
detector = fit_anomaly_detector(featured, contamination=0.03)
scored = featured.copy()
scored["predicted_performance_score"] = performance_model.predict(scored[FEATURES])
scored = apply_anomaly_detector(detector, scored)
alerts = generate_alerts(scored)
print("Anomaly rate:", scored.is_anomaly.mean(), "Alerts:", len(alerts))
display(alerts.head())


In [ ]:
# 5. Visualize sensor timeline and anomaly flags
sample_athlete = scored.athlete_id.iloc[0]
sample = scored[scored.athlete_id == sample_athlete]
plt.figure(figsize=(13,4))
plt.plot(sample.timestamp, sample.heart_rate, label="Heart rate")
flagged = sample[sample.is_anomaly]
plt.scatter(flagged.timestamp, flagged.heart_rate, label="Anomaly flag", marker="x")
plt.title(f"Synthetic sensor stream — {sample_athlete}")
plt.xlabel("Timestamp"); plt.ylabel("Heart rate (bpm)"); plt.legend(); plt.grid(alpha=.2); plt.show()


In [ ]:
# 6. Save demo outputs
pd.DataFrame([metrics]).to_csv("results/performance_metrics.csv", index=False)
scored.to_csv("results/athlete_analytics_output.csv", index=False)
alerts.to_csv("results/alert_records.csv", index=False)
print("Saved output files in results/. Synthetic demo only.")


## Reporting checklist
- Clearly label the data as synthetic if using the built-in generator.
- Document sampling frequency, sensor units, and athlete consent for real data.
- Define how performance_score was measured.
- Do not interpret Isolation Forest flags as injury diagnoses.
- Report model metrics only for the exact split and data used.